# NB07 — Atlas: ConvNeXt, ViT & MLP-Mixer


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        ## What this notebook runs

        | | |
        |---|---|
        | **Architectures** | 3 |
        | **Seeds each** | 3 |
        | **Total training runs** | **9** |
        | **Estimated GPU time** | **~36 GPU-hours** |

        ### Wall-clock, by how many accounts you run

        | NUM_WORKERS | Wall-clock | Kaggle sessions each |
        |---|---|---|
        | 1 | ~36.4 h | 5 |
        | 2 | ~19.1 h | 3 |
        | 4 | ~10.7 h | 2 |
        | 6 | ~6.9 h | 1 |

        ### Per architecture

        | Architecture | Epochs | Est. per run | × seeds | Basis |
        |---|---|---|---|---|
        | `convnext_femto` | 300 | 4.16 h | 12.5 h | estimate |
        | `mixer_nano` | 300 | 2.77 h | 8.3 h | estimate |
        | `vit_tiny` | 300 | 5.20 h | 15.6 h | estimate |

        > Estimates are calibrated against measured Phase 0 timings on a Kaggle T4 (8.32 s per cost-unit-epoch). 0 of 3 architectures here are measured; the rest are priced from a relative cost model that self-corrects as runs finish.

        > A Kaggle session lasts ~9–12 h and this pipeline pauses cleanly at 8.5 h, so a run needing more than one session resumes automatically — just start a fresh session and re-run.

        ## Why this group is its own notebook

        **These are the most important architectures in the project and the reason Q3 is interesting at all.** A Vision Transformer processes an image in a fundamentally different way from a ResNet, and an MLP-Mixer has almost no built-in assumption about images being spatial. Our central prediction is that agreement drops when we cross that boundary. Without these three, the transfer study is 'do ResNets agree with other ResNets', which is a much weaker paper.

        They're in a separate notebook because they need a **completely different training recipe** — AdamW, long warmup, strong augmentation, label smoothing. Under plain SGD they don't learn at all; accuracy sits near 1% forever. The library selects the right recipe automatically, but keeping them separate means a failure here doesn't take the CNNs with it.

        ## Why 3 seeds

        A single training run's accuracy is partly luck. Reporting one number
        without a spread isn't publishable — and more importantly here, the
        seed-to-seed comparison IS the noise ceiling that every transfer number
        in the paper gets divided by.

        ## Architectures in this notebook

        `convnext_femto`, `vit_tiny`, `mixer_nano`

        ## Running this across accounts

        Set `NUM_WORKERS` to how many accounts you have and give each a different
        `WORKER_ID`. The work splits by estimated GPU-hours, not by run count, so
        every account should finish at roughly the same time even though the
        models differ in cost.

        **This is long-running.** It pushes to HuggingFace every 30 minutes,
        pauses cleanly at 8.5 hours, and resumes exactly where it stopped when
        you start a fresh session and re-run.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   b4c1b3574dd2   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  6abdba4ff104   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Session

In [ ]:
# === Who am I? =============================================================
#
# THIS NOTEBOOK: 9 run(s), ~36 GPU-hours total.
# At NUM_WORKERS = 1 that is all of it on this account.
# Raise NUM_WORKERS and run the same notebook on each account
# with a different WORKER_ID to divide it.
#
# ACCOUNT   labels this Kaggle account in the run log. Two accounts calling
#           themselves the same thing makes the log useless.
# WORKER_ID splits the work. Every account runs THE SAME notebook; the only
#           thing that differs is this number. Account 1 -> 0, account 2 -> 1,
#           and so on. Each account then works out, by pure arithmetic, which
#           jobs belong to it -- no communication needed, no chance of two
#           accounts training the same model, no chance of a job being missed.
#
# DEFAULT IS 1: this account does everything in this notebook. That is the
# simplest thing that works. Change it only when you actually have several
# accounts running at once.
ACCOUNT     = 'acct1'      # <<< CHANGE ME
NUM_WORKERS = 1          # <<< how many accounts you are running in parallel
WORKER_ID   = 0            # <<< CHANGE ME: 0, 1, 2, ... up to NUM_WORKERS-1

sess = msc.Session(account=ACCOUNT, phase='p1', dataset='cifar100',
                   worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                   shard_mode='cost',        # balance GPU-hours, not job counts
                   enable_hf=True,
                   session_limit_h=8.5,      # push + pause before Kaggle kills us
                   commits_per_hour_limit=20,# 6 accounts x 20 = 120 < HF's 128/hr
                   batch_interval_sec=1800)  # the 30-minute push policy

## Step 2 — Dataset

In [ ]:
# === Get CIFAR-100 =========================================================
# Looks in this order: attached Kaggle dataset (instant) -> earlier extraction
# -> Kaggle CLI download -> torchvision as a last resort.
# Everything lands in /kaggle/temp (~1 TB scratch), never in /kaggle/working
# (20 GB, and that is the space your results need).
DATA_ROOT = sess.prepare_data()
print('dataset:', DATA_ROOT)

## Step 3 — Catch up

In [ ]:
# === Catch up with what has already been done ==============================
# Downloads only what this notebook needs -- never the whole repo, which would
# fill the 20 GB disk instantly.
#
# It also rebuilds the progress record from the actual training logs instead of
# trusting the status file. If a session died between writing a log and pushing
# its status, those two disagree, and the log is the one that tells the truth.
# Runs marked "finished" that clearly are not get reset so they resume.
sess.sync_state(verbose=True)
sess.status()

## Step 4 — See the plan and the time before committing to it

Prints how long this will actually take, using measured per-epoch times
from any runs already finished. If the split looks badly unbalanced, or
the hours are more than you want on one account, change `NUM_WORKERS`
now rather than discovering it on day three.

In [ ]:
ARCHS = ['convnext_femto', 'vit_tiny', 'mixer_nano']
SEEDS = (1, 2, 3)

cfgs = [sess.config(a, seed=s) for a in ARCHS for s in SEEDS]
run_ids = [c['run_id'] for c in cfgs]

In [ ]:
# === How long will this take? ==============================================
# Recomputed live, using measured per-epoch times from any runs that have
# already finished. The more of the project you have completed, the more
# accurate this gets -- the built-in estimates are only a starting prior.
est = msc.estimate_phase(run_ids, num_workers=NUM_WORKERS)
print(f"runs in this notebook : {est['n_runs']}")
print(f"total GPU time        : ~{est['total_gpu_hours']:.1f} GPU-hours")
print(f"at NUM_WORKERS={NUM_WORKERS:<2d}      : ~{est['wall_clock_hours']:.1f} h wall-clock "
      f"({est['sessions_needed']} Kaggle session(s) on this account)")
print(f"cost model            : {est['frac_measured']:.0%} of these architectures "
      f"have measured timings; the rest are estimates")
if NUM_WORKERS > 1:
    print(f"per-worker hours      : "
          f"{[round(h,1) for h in est['per_worker_hours']]}")
print()
for nw in (1, 2, 4, 6):
    if nw > est['n_runs']:
        break
    e = msc.estimate_phase(run_ids, num_workers=nw)
    mark = '  <-- you' if nw == NUM_WORKERS else ''
    print(f"  NUM_WORKERS={nw}: ~{e['wall_clock_hours']:5.1f} h wall-clock{mark}")

In [ ]:
display(msc.shard_report(run_ids, NUM_WORKERS, mode='cost'))
plan = sess.plan(run_ids, title='NB07 plan')

## Step 5 — Train

Safe to stop at any moment. Re-run in a fresh session to continue.

In [ ]:
summaries = sess.run_all(cfgs, title='atlas training')

import pandas as pd
pd.DataFrame([{k: s.get(k) for k in
               ('run_id', 'arch', 'seed', 'best_accuracy', 'reference_accuracy',
                'accuracy_gap_vs_reference', 'recipe_ok', 'num_epochs_run',
                'total_energy_kwh')}
              for s in summaries if s.get('status') == 'completed'])

## Step 6 — Progress across ALL accounts

Not just this one — this reads the shared record on HuggingFace, so you
can see how the whole team is doing.

In [ ]:
import pandas as pd
want = set(run_ids)
state = sess.registry.latest()
rows = [{'run_id': r, 'state': state.get(r, {}).get('state', 'not started'),
         'accuracy': state.get(r, {}).get('best_accuracy'),
         'by': state.get(r, {}).get('account'),
         'mine': r in plan.mine} for r in sorted(want)]
prog = pd.DataFrame(rows)
display(prog)
n_done = int((prog.state == 'completed').sum())
print(f'\n{n_done} of {len(prog)} runs finished across all accounts '
      f'({n_done/max(1,len(prog))*100:.0f}%)')

## Step 7 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

# D-19: draining the upload queue is NOT the same as the files being on
# HuggingFace, and "[SESSION] done" reads like a confirmation it is not.
# Ask the repository before you close this tab.
#
# D-20: three states, not two. FINISHED and RESUMABLE are both safe -- a run
# paused at epoch 120 whose ckpt_last.pt is on HF loses nothing when you close
# the tab. Only AT RISK (no summary.json AND no checkpoint) needs action.
try:
    _ids = [c['run_id'] for c in cfgs]
except NameError:
    _ids = []
if _ids:
    sess.confirm_on_hf(_ids)